<div align="center">

# EDA 

</div>


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv(r'X:\nasim_xhqpjmy\Code\MLops\Race-Telemetry\dataset\data.csv')
df.head(20)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum().sum()

<h4 align="center">Core Telemetry Distribution</h4>


In [ ]:
cols = ['speed', 'current_engine_rpm', 'torque']
df[cols].hist(bins=50, figsize=(12,6))
plt.suptitle("Core Telemetry Distributions", fontsize=14)
plt.show()

<h4 align="center">Correlation Analysis</h4>


In [ ]:
corr = df[['speed','power','torque','gear','steer','acceleration','brake']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')

<h4 align="center">Speed Vs Gear Over Time</h4>


In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df['timestamp_ms'], df['speed'], label='Speed (km/h)')
plt.plot(df['timestamp_ms'], df['gear']*30, label='Gear ×30', alpha=0.7)
plt.legend()
plt.xlabel('Timestamp (ms)')
plt.title('Speed vs Gear Over Time')
plt.show()

<h4 align="center">Tire Slip Over Time (For Front and Rear Tires)</h4>


In [ ]:
slip_cols = [col for col in df.columns if "tire_slip_angle" in col]
plt.figure(figsize=(10,5))
df[slip_cols].plot(kind='line', alpha=0.6)
plt.title("Tire Slip Angle over Time (Front & Rear)")
plt.xlabel("Frame Index")
plt.ylabel("Slip Angle (°)")
plt.show()

<h4 align="center">Calculating the Fastest Laps</h4>


In [ ]:
number_of_laps = max(df['lap_number']) + 1 # They are 0 indexed

best_lap = 0
worst_lap = 0

latest_best_time = 0
latest_worst_time = 0

for lap in range(0, number_of_laps):
    lap_data = df[df['lap_number'] == lap]
    time_value_in_lap = max(lap_data['current_lap_time'])
    print(f'Lap {lap} - {time_value_in_lap}')
    
    if latest_best_time < time_value_in_lap:
        best_lap = lap
        
    if latest_worst_time > time_value_in_lap:
        worst_lap = lap
        
best_lap, worst_lap

<h4 align="center">Throttle vs Braking vs Speed (m/s) vs Boost (psi)</h4>


In [ ]:
throttle = lap_data['acceleration'] / 5.0
braking = lap_data['brake'] / 4.5
speed = lap_data['speed']
boost = lap_data['boost']
time = lap_data.index

# Plot telemetry
plt.figure(figsize=(10, 5))
plt.plot(time, throttle, label='Throttle', linewidth=1.5)
plt.plot(time, braking, label='Braking', color='#a2afba', linewidth=1.5)
plt.plot(time, speed, label='Speed (m/s)', color='#ff00ff', linewidth=1.5)
plt.plot(time, boost, label='Boost (psi)', color='black', linewidth=1.5)

# Formatting
plt.title('Throttle vs Braking vs Speed (m/s) vs Boost (psi)')
plt.xlabel('Time')
plt.ylabel('Amount')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

<h4 align="center">Engine RPM</h4>


In [ ]:
rpm_data = lap_data['current_engine_rpm']
time = lap_data.index

# Plot Engine RPM
plt.figure(figsize=(10, 5))
plt.plot(time, rpm_data, color='tab:red', linewidth=1.5)

# Formatting
plt.title('Engine RPM')
plt.xlabel('Time')
plt.ylabel('RPM')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

<h4 align="center">Speed (m/s) vs Boost (psi)</h4>


In [ ]:
speed_data = lap_data['speed']
boost_data = lap_data['boost']
time = lap_data.index

# Plot speed and boost together
plt.figure(figsize=(10, 5))
plt.plot(time, speed_data, label='Speed (m/s)', color='tab:blue', linewidth=1.5)
plt.plot(time, boost_data, label='Boost (psi)', color='tab:orange', linewidth=1.5)

# Formatting
plt.title('Speed (m/s) vs Boost (psi)')
plt.xlabel('Time')
plt.ylabel('Amount')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


<h4 align="center">Power vs Torque</h4>


In [ ]:
torque_data = lap_data['torque'] / 1.356   # Nm → ft-lb
power_data = lap_data['power'] / 745.7     # W → HP
time = lap_data.index

# Plot Power vs Torque
plt.figure(figsize=(10, 5))
plt.plot(time, torque_data, label='Torque (ft-lb)', color='green', linewidth=1.5)
plt.plot(time, power_data, label='Power (HP)', color='tab:red', linewidth=1.5)

# Formatting
plt.title('Power vs Torque')
plt.xlabel('Time')
plt.ylabel('Amount')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


<h4 align="center">Tire Temperature</h4>


In [ ]:
mask = lap_data.columns.str.contains('tire_temp_.*')
tire_temps = lap_data.loc[:, mask]

# Convert average tire temperature from Fahrenheit to Celsius
tire_temps_c = (tire_temps.mean(axis=1) - 32) * 5 / 9
time = lap_data.index

# Plot average tire temperature
plt.figure(figsize=(10, 5))
plt.plot(time, tire_temps_c, color='tab:orange', linewidth=1.5)

# Formatting
plt.title('Tire Temperature')
plt.xlabel('Time')
plt.ylabel('Temperature (°C)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

<h4 align="center">Throttle vs Brake Zones on Track Map</h4>


In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(
    x='position_x', y='position_y', 
    hue=np.where(df['brake']>0, 'Brake', 'Throttle'), 
    palette={'Brake':'red','Throttle':'green'},
    s=3, alpha=0.6, data=df
)
plt.title("Throttle vs Brake Zones on Track Map")
plt.legend()
plt.show()

<h4 align="center">Gear Usage Frequency</h4>


In [ ]:
plt.figure(figsize=(8,4))
sns.countplot(x='gear', data=df, palette='viridis')
plt.title("Gear Usage Frequency")
plt.xlabel("Gear")
plt.ylabel("Count")
plt.show()

<h4 align="center">Car Path Colored by Speed</h4>


In [ ]:
plt.figure(figsize=(8,8))
plt.scatter(df['position_x'], df['position_y'], c=df['speed'], cmap='viridis', s=1)
plt.colorbar(label='Speed')
plt.title('Car Path Colored by Speed')
plt.xlabel('Position X')
plt.ylabel('Position Y')
plt.show()

<h4 align="center">Driver Input Over Time</h4>


In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

axs[0].plot(df['timestamp_ms'], df['acceleration'], color='g', label='Acceleration')
axs[1].plot(df['timestamp_ms'], df['brake'], color='r', label='Brake')
axs[2].plot(df['timestamp_ms'], df['steer'], color='b', label='Steer Angle')

for ax in axs:
    ax.legend()
    ax.grid(True)
plt.xlabel('Timestamp (ms)')
plt.suptitle('Driver Input Over Time')
plt.show()

<h4 align="center">Average Wheel Speed Distribution</h4>


In [ ]:
df['avg_wheel_speed'] = df[['wheel_rotation_speed_front_left',
                             'wheel_rotation_speed_front_right',
                             'wheel_rotation_speed_rear_left',
                             'wheel_rotation_speed_rear_right']].mean(axis=1)
df['avg_wheel_speed'].hist(bins=50, figsize=(10,5))
plt.title('Average Wheel Speed Distribution')
plt.xlabel('Average Wheel Speed')
plt.ylabel('Frequency')
plt.show()